# Summary of Insights
## Sentiment and Topic Trajectories Across Crisis Phases

This notebook consolidates findings from sentiment analysis, topic modeling, and cross-phase analysis into a single narrative summary. It covers:

1. **Sentiment Trajectory Across Phases** — how polarity and negativity evolve chronologically
2. **Phases Ranked by Negativity** — which crisis events provoked the most negative response
3. **Day 1 vs Rest: Within-Phase Outrage Decay** — whether initial outrage decays or sustains within each phase
4. **Topics Per Phase** — dominant topics per phase and their sentiment valence
5. **Topic Trajectory Across Phases** — how topics persist or shift across phase boundaries
6. **Additional Analyses** — phase boundary sentiment shifts

## Setup
Import dependencies, configure paths, and define phase constants.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import chi2_contingency, kendalltau

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 5)

SEED = 42
np.random.seed(SEED)

OUTPUT_DIR = Path("../../02-analysis/output")
FIGURES_DIR = Path("../../outputs/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

PHASE_ORDER = [
    "01-the_incident",
    "02-autopsy_request",
    "03-public_outcry",
    "03.1-baldwin_apology",
    "04-scholarships_withdrawal",
    "05-resignation_presscon",
    "06-townhall_podcast",
    "07-uaap_ban",
    "08-legal_probe",
]

PHASE_LABELS = [
    "The Incident",
    "Autopsy Request",
    "Public Outcry",
    "Baldwin Apology",
    "Scholarships Withdrawn",
    "Resignation Press Con",
    "Townhall Podcast",
    "UAAP Ban",
    "Legal Probe",
]

PHASE_DISPLAY = dict(zip(PHASE_ORDER, PHASE_LABELS))

### Crisis Event Timeline
Reference table anchoring each phase to actual news events.

In [ ]:
EVENT_TIMELINE = [
    ("01-the_incident",           "Jun 8",  "Drowning incident; Ateneo confirms deaths"),
    ("02-autopsy_request",        "Jun 9",  "Family requests autopsy; NBI/CIDG coordinate"),
    ("03-public_outcry",           "Jun 11", "Viral rumors: ankle weights, hazing"),
    ("03.1-baldwin_apology",       "Jun 12", "Baldwin video apology; remains arrive"),
    ("04-scholarships_withdrawal", "Jun 14", "Scholarships offered; Filoil withdrawal"),
    ("05-resignation_presscon",    "Jun 15", "Fr. Yap accepts Baldwin's resignation"),
    ("06-townhall_podcast",        "Jun 16", "Team captain townhall; podcast interviews"),
    ("07-uaap_ban",                "Jun 19", "UAAP bans coaching staff"),
    ("08-legal_probe",             "Jun 19", "CIDG/NBI hearings; hazing investigation"),
]

EVENT_DATES = pd.DataFrame(EVENT_TIMELINE, columns=["phase", "event_date_str", "event_description"])
EVENT_DATES["event_date_num"] = pd.to_datetime(EVENT_DATES["event_date_str"], format="%b %d")
phase_boundaries = EVENT_DATES.copy()
phase_boundaries["label"] = phase_boundaries["phase"].map(PHASE_DISPLAY)

print("Phase boundary dates:")
for _, row in phase_boundaries.iterrows():
    print(f"  {row['event_date_str']:12s} | {row['label']:25s} | {row['event_description']}")

## Load & Merge Data
Load all scored sentiment files and topic-modeled CSVs, merging into a single DataFrame.

In [ ]:
frames = []
for phase in PHASE_ORDER:
    path = OUTPUT_DIR / f"scored_{phase}.csv"
    if not path.exists():
        print(f"WARNING: {path.name} not found -- skipping")
        continue
    df_batch = pd.read_csv(path, parse_dates=["create_time"])
    df_batch["phase"] = phase
    df_batch["phase_label"] = PHASE_DISPLAY[phase]
    frames.append(df_batch)

df_sentiment = pd.concat(frames, ignore_index=True)
print(f"Loaded sentiment data: {len(df_sentiment):,} comments across {df_sentiment['phase'].nunique()} phases")

In [ ]:
topic_frames = []
for path in sorted(OUTPUT_DIR.glob("topics_*.csv")):
    tt = pd.read_csv(path)
    phase = path.stem.replace("topics_", "")
    tt["phase"] = phase
    topic_frames.append(tt)

df_topics = pd.concat(topic_frames, ignore_index=True)
topic_cols = [c for c in df_topics.columns if c in ("topic_id", "topic_label")]
keep_cols = ["text", "phase"] + topic_cols
keep_cols = [c for c in keep_cols if c in df_topics.columns]
df_topics_sub = df_topics[keep_cols].copy()

df = df_sentiment.merge(df_topics_sub, on=["text", "phase"], how="left", suffixes=("", "_topic"))

print(f"Merged DataFrame: {len(df):,} rows, {len(df.columns)} columns")
print(f"Columns: {list(df.columns)}")

In [ ]:
print(f"Rows with missing sentiment_label: {df['sentiment_label'].isna().sum()}")
print(f"Rows with missing topic_label: {df['topic_label'].isna().sum()}")
print()
print(f"Timestamp range: {df['create_time'].min()} to {df['create_time'].max()}")
print(f"Unique dates: {df['create_time'].dt.date.nunique()}")
print()
print("Comments per phase:")
for p in PHASE_ORDER:
    n = (df['phase'] == p).sum()
    if n > 0:
        print(f"  {PHASE_DISPLAY[p]:35s} {n:5,}")

> **Note on missing topic labels:** BERTopic's HDBSCAN clustering assigned
> 31.9% of comments (3,382 rows) to the noise cluster (`topic_id = -1`).
> These comments did not form dense enough clusters for coherent topic assignment.
> All topic-per-phase analyses reflect only the 68.1% of comments with assigned topics.
> This noise rate is within normal range for BERTopic on informal code-switched
> social media text.

---
## 1. Sentiment Trajectory Across Phases

How does public sentiment evolve as the crisis unfolds? We aggregate sentiment per phase and plot the **Polarity Index** (%positive − %negative) in chronological order. A more negative value means more outrage relative to positive sentiment. We also compute a **Mann-Kendall trend test** to check if polarity shows a monotonic trend over time.

In [ ]:
phase_agg = (
    df.groupby("phase")
    .agg(
        total_comments=("sentiment_label", "count"),
        n_negative=("sentiment_label", lambda x: (x == "NEGATIVE").sum()),
        n_positive=("sentiment_label", lambda x: (x == "POSITIVE").sum()),
        n_neutral=("sentiment_label", lambda x: (x == "NEUTRAL").sum()),
        mean_likes=("like_count", "mean"),
    )
    .reset_index()
)

phase_agg["pct_negative"] = phase_agg["n_negative"] / phase_agg["total_comments"] * 100
phase_agg["pct_positive"] = phase_agg["n_positive"] / phase_agg["total_comments"] * 100
phase_agg["pct_neutral"] = phase_agg["n_neutral"] / phase_agg["total_comments"] * 100
phase_agg["polarity_index"] = phase_agg["pct_positive"] - phase_agg["pct_negative"]
phase_agg["label"] = phase_agg["phase"].map(PHASE_DISPLAY)
phase_agg["event_date"] = phase_agg["phase"].map(
    dict(zip(EVENT_DATES["phase"], EVENT_DATES["event_date_str"]))
)

phase_agg["phase_num"] = phase_agg["phase"].map(
    {p: i for i, p in enumerate(PHASE_ORDER)}
)
phase_agg = phase_agg.sort_values("phase_num").reset_index(drop=True)

print("Sentiment per phase (sorted chronologically):")
display(phase_agg[["label", "event_date", "total_comments",
                    "pct_negative", "pct_positive", "pct_neutral",
                    "polarity_index", "mean_likes"]].round(1))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

x = np.arange(len(phase_agg))
ax1.plot(x, phase_agg["polarity_index"], color="crimson", linewidth=2.5,
         marker="o", markersize=8, zorder=3)
ax1.fill_between(x, phase_agg["polarity_index"], 0,
                 where=(phase_agg["polarity_index"] < 0),
                 color="crimson", alpha=0.15)
ax1.axhline(y=0, color="black", linewidth=0.5)
ax1.set_xticks(x)
ax1.set_xticklabels(phase_agg["label"], rotation=45, ha="right", fontsize=9)
ax1.set_ylabel("Polarity Index (%positive - %negative)")
ax1.set_title("Polarity Trajectory Across Phases (chronological)")

for _, row in phase_agg.iterrows():
    ax1.annotate(f"{row['pct_negative']:.0f}% neg",
                 (row["phase_num"], row["polarity_index"]),
                 textcoords="offset points", xytext=(0, -15),
                 ha="center", fontsize=8, color="darkred")

phase_sorted = phase_agg.sort_values("pct_negative", ascending=True)
colors = plt.cm.RdYlGn_r(phase_sorted["pct_negative"] / 100)
bars = ax2.barh(phase_sorted["label"], phase_sorted["pct_negative"],
                color=colors, edgecolor="gray", linewidth=0.5)

for bar, val, pol in zip(bars, phase_sorted["pct_negative"], phase_sorted["polarity_index"]):
    ax2.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
             f"{val:.1f}%  (polarity: {pol:.1f})",
             va="center", fontsize=8)

ax2.set_xlabel("% Negative")
ax2.set_title("Phases Ranked by Negativity")
ax2.set_xlim(0, 100)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "summary_phase_sentiment_trajectory.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
most_neg = phase_agg.loc[phase_agg["pct_negative"].idxmax()]
least_neg = phase_agg.loc[phase_agg["pct_negative"].idxmin()]

print(f"Most negative phase:  {most_neg['label']}  ({most_neg['pct_negative']:.1f}% negative, "
      f"polarity {most_neg['polarity_index']:.1f})")
print(f"Least negative phase: {least_neg['label']}  ({least_neg['pct_negative']:.1f}% negative, "
      f"polarity {least_neg['polarity_index']:.1f})")
print()
print(f"Polarity range: {phase_agg['polarity_index'].min():.1f} to "
      f"{phase_agg['polarity_index'].max():.1f} "
      f"(delta = {phase_agg['polarity_index'].max() - phase_agg['polarity_index'].min():.1f})")
print()

weighted_pol = (phase_agg['polarity_index'] * phase_agg['total_comments']).sum() / phase_agg['total_comments'].sum()
print(f"Volume-weighted avg polarity: {weighted_pol:.1f}")
print()

# Mann-Kendall trend test on chronological polarity
tau, p_value = kendalltau(phase_agg["polarity_index"], np.arange(len(phase_agg)))
print(f"Mann-Kendall trend test: tau={tau:.4f}, p={p_value:.4f}")
if p_value < 0.05:
    direction = "improving" if tau > 0 else "worsening"
    print(f"  Statistically significant {direction} trend (p < 0.05).")
else:
    print(f"  No significant monotonic trend. Sentiment fluctuates without consistent direction.")

---
## 2. Phases Ranked by Negativity

Ranking phases by percentage of negative comments reveals which crisis events provoked the most outrage. The horizontal bar chart above (right panel) already shows this ranking. Below is the full breakdown.

In [ ]:
ranking = phase_agg.sort_values("pct_negative", ascending=False).reset_index(drop=True)
ranking["rank"] = range(1, len(ranking) + 1)
display(
    ranking[["rank", "label", "event_date", "total_comments",
             "pct_negative", "pct_positive", "pct_neutral", "polarity_index"]]
    .round(1)
    .rename(columns={"label": "Phase"})
)

---
## 3. Day 1 vs Rest: Within-Phase Outrage Decay

### What this measures

For each crisis phase, we split comments into two windows:
- **Day 1** — comments posted on the first calendar date of that phase (when the revelation breaks)
- **Rest** — all subsequent comments within that phase

This tests whether **initial outrage decays** as a phase unfolds. If %negative is higher on Day 1 than Rest, people gradually calm down. If %negative stays the same or rises, outrage sustains or even intensifies.

We use a **chi-square test of independence** to check if the sentiment distribution (Neg/Pos/Neu) differs significantly between Day 1 and Rest, and report **Cramer's V** as the effect size.

In [ ]:
phase_day1 = []
for phase in df["phase"].unique():
    phase_df = df[df["phase"] == phase].copy()
    phase_df["date"] = phase_df["create_time"].dt.date
    earliest = phase_df["date"].min()

    day1 = phase_df[phase_df["date"] == earliest]
    rest = phase_df[phase_df["date"] > earliest]

    def sentiment_dist(subset, label):
        neg = (subset["sentiment_label"] == "NEGATIVE").sum()
        pos = (subset["sentiment_label"] == "POSITIVE").sum()
        neu = (subset["sentiment_label"] == "NEUTRAL").sum()
        total = len(subset)
        return {
            "phase": phase,
            "label": PHASE_DISPLAY[phase],
            "window": label,
            "total": total,
            "pct_neg": neg / total * 100 if total else 0,
            "pct_pos": pos / total * 100 if total else 0,
            "pct_neu": neu / total * 100 if total else 0,
            "polarity": (pos - neg) / total * 100 if total else 0,
        }

    phase_day1.append(sentiment_dist(day1, "Day 1"))
    if len(rest) > 0:
        phase_day1.append(sentiment_dist(rest, "Rest"))

day1_df = pd.DataFrame(phase_day1)
day1_df["phase_num"] = day1_df["phase"].map(
    {p: i for i, p in enumerate(PHASE_ORDER)}
)
day1_df = day1_df.sort_values(["phase_num", "window"]).reset_index(drop=True)

display(
    day1_df[["label", "window", "total", "pct_neg", "pct_pos", "pct_neu", "polarity"]].round(1)
)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for idx, phase in enumerate(PHASE_ORDER):
    ax = axes[idx]
    pair = day1_df[day1_df["phase"] == phase]
    if len(pair) == 0:
        ax.set_visible(False)
        continue

    x = np.arange(3)
    width = 0.3

    for i, w in enumerate(["Day 1", "Rest"]):
        row = pair[pair["window"] == w]
        if len(row) == 0:
            continue
        vals = [row["pct_neg"].values[0], row["pct_pos"].values[0], row["pct_neu"].values[0]]
        ax.bar(x + i * width, vals, width, label=w if idx == 0 else None)

    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(["Neg", "Pos", "Neu"])
    ax.set_title(PHASE_DISPLAY[phase], fontsize=10)
    ax.set_ylim(0, 100)
    if idx == 0:
        ax.legend(fontsize=8)

fig.suptitle("Within-Phase Sentiment: Day 1 vs Rest", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "summary_day1_vs_rest_faceted.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("Within-phase comparison: Day 1 vs Rest")
header = f"{'Phase':30s} {'Day1 N':>6s} {'Rest N':>6s} {'Chi2':>8s} {'p':>8s} {'Cramer V':>8s} {'Sig?':>6s}"
print(header)
print("-" * 75)

results = []
for phase in day1_df["phase"].unique():
    pair = day1_df[day1_df["phase"] == phase]
    if len(pair) < 2:
        continue

    day1_row = pair[pair["window"] == "Day 1"].iloc[0]
    rest_row = pair[pair["window"] == "Rest"].iloc[0]

    contingency = np.array([
        [day1_row["pct_neg"] * day1_row["total"] / 100,
         day1_row["pct_pos"] * day1_row["total"] / 100,
         day1_row["pct_neu"] * day1_row["total"] / 100],
        [rest_row["pct_neg"] * rest_row["total"] / 100,
         rest_row["pct_pos"] * rest_row["total"] / 100,
         rest_row["pct_neu"] * rest_row["total"] / 100],
    ])

    chi2, p, _, _ = chi2_contingency(contingency)
    n = contingency.sum()
    cramer_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))

    sig = "YES" if p < 0.05 else "no"
    results.append({
        "phase": phase,
        "label": PHASE_DISPLAY[phase],
        "day1_n": int(day1_row["total"]),
        "rest_n": int(rest_row["total"]),
        "delta_neg": rest_row["pct_neg"] - day1_row["pct_neg"],
        "delta_polarity": rest_row["polarity"] - day1_row["polarity"],
        "chi2": chi2,
        "p": p,
        "cramer_v": cramer_v,
        "significant": sig,
    })

    print(
        f"{PHASE_DISPLAY[phase]:30s} {int(day1_row['total']):6d} "
        f"{int(rest_row['total']):6d} {chi2:8.2f} {p:8.4f} {cramer_v:8.4f} {sig:>6s}"
    )

results_df = pd.DataFrame(results)
print()
print("Delta interpretation:")
print("  delta_neg = Rest %neg - Day1 %neg (positive = outrage decays)")
print("  delta_polarity = Rest polarity - Day1 polarity (positive = sentiment improves)")

In [ ]:
sig_count = (results_df["p"] < 0.05).sum()
print(f"Phases with significant Day 1 / Rest difference: {sig_count}/{len(results_df)}")
print()
print("Key finding: Only the Townhall Podcast phase shows a statistically significant")
print("shift between Day 1 and Rest (p=0.0013), and the effect size is small (Cramer's V=0.051).")
print("For all other phases, within-phase sentiment is stable — initial outrage neither decays")
print("nor intensifies significantly.")
print()
print("Largest magnitude shifts (though not statistically significant):")
results_df["abs_delta_neg"] = results_df["delta_neg"].abs()
shifted = results_df.sort_values("abs_delta_neg", ascending=False)
for _, row in shifted.iterrows():
    arrow = chr(8593) if row["delta_neg"] > 0 else chr(8595)
    print(
        f"  {row['label']:30s} {arrow} {abs(row['delta_neg']):.1f}pp neg "
        f"(p={row['p']:.4f})"
    )

---
## 4. Topics Per Phase

For each phase, we identify the top-3 most prevalent topics and compute their sentiment polarity. This reveals what people were talking about during each crisis event and whether certain themes carry more outrage than others.

In [ ]:
topic_phases = df.dropna(subset=["topic_label"]).copy()

top_topics = []
for phase in PHASE_ORDER:
    phase_topics = topic_phases[topic_phases["phase"] == phase]
    total = len(phase_topics)
    if total == 0:
        continue

    topic_counts = phase_topics["topic_label"].value_counts()
    for rank, (topic, count) in enumerate(topic_counts.head(3).items(), 1):
        pct = count / total * 100
        topic_sub = phase_topics[phase_topics["topic_label"] == topic]
        neg = (topic_sub["sentiment_label"] == "NEGATIVE").sum()
        pos = (topic_sub["sentiment_label"] == "POSITIVE").sum()
        pol = (pos - neg) / count * 100 if count else 0
        top_topics.append({
            "phase": PHASE_DISPLAY[phase], "rank": rank, "topic": topic,
            "n": count, "pct": pct, "polarity": pol,
        })

top3_df = pd.DataFrame(top_topics)
display(top3_df[["phase", "rank", "topic", "n", "pct", "polarity"]].round(1))

In [ ]:
topic_polarity = (
    topic_phases
    .groupby(["topic_label", "phase"])
    .apply(
        lambda x: ((x["sentiment_label"] == "POSITIVE").sum()
                   - (x["sentiment_label"] == "NEGATIVE").sum())
                  / len(x) * 100
    )
    .reset_index(name="polarity")
)

topic_pivot = topic_polarity.pivot(
    index="topic_label", columns="phase", values="polarity"
)
available = [c for c in PHASE_ORDER if c in topic_pivot.columns]
topic_pivot = topic_pivot[available]
topic_pivot.columns = [PHASE_DISPLAY[c] for c in topic_pivot.columns]

topic_pivot = topic_pivot.dropna(thresh=2)
topic_pivot = topic_pivot.fillna(0)

fig, ax = plt.subplots(figsize=(12, max(6, len(topic_pivot) * 0.4)))
sns.heatmap(topic_pivot, cmap="RdBu_r", center=0, annot=True, fmt=".0f",
            linewidths=0.5, ax=ax, cbar_kws={"label": "Polarity Index"})
ax.set_title("Topic Sentiment Polarity by Phase")
ax.set_xlabel("Crisis Phase")
ax.set_ylabel("Topic")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "summary_topic_phase_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
topic_overall = (
    topic_phases.groupby("topic_label")
    .agg(
        total_comments=("sentiment_label", "count"),
        n_neg=("sentiment_label", lambda x: (x == "NEGATIVE").sum()),
        n_pos=("sentiment_label", lambda x: (x == "POSITIVE").sum()),
    )
    .reset_index()
)
topic_overall = topic_overall[topic_overall["total_comments"] >= 10].copy()
topic_overall["pct_neg"] = topic_overall["n_neg"] / topic_overall["total_comments"] * 100
topic_overall["pct_pos"] = topic_overall["n_pos"] / topic_overall["total_comments"] * 100
topic_overall["polarity"] = topic_overall["pct_pos"] - topic_overall["pct_neg"]

most_angry = topic_overall.sort_values("pct_neg", ascending=False).head(5)
most_positive = topic_overall.sort_values("pct_pos", ascending=False).head(5)

print("Most negative topics (top 5):")
for _, row in most_angry.iterrows():
    print(f"  {row['topic_label']:40s} {row['pct_neg']:5.1f}% neg  ({int(row['total_comments']):4d} comments)")
print()
print("Most positive topics (top 5):")
for _, row in most_positive.iterrows():
    print(f"  {row['topic_label']:40s} {row['pct_pos']:5.1f}% pos  ({int(row['total_comments']):4d} comments)")

---
## 5. Topic Trajectory Across Phases

How do topics evolve as the crisis progresses? We track which topics appear in multiple phases (persistent) vs. those unique to a single phase. We also identify the dominant topic at each phase to see how the conversation shifts at each new revelation.

In [ ]:
topic_phase_count = (
    topic_phases.groupby("topic_label")["phase"]
    .nunique()
    .reset_index(name="n_phases")
    .sort_values("n_phases", ascending=False)
)

persistent = topic_phase_count[topic_phase_count["n_phases"] > 1]
phase_specific = topic_phase_count[topic_phase_count["n_phases"] == 1]

print(f"Topics appearing in multiple phases (persistent): {len(persistent)}")
for _, row in persistent.iterrows():
    print(f"  {row['topic_label']:40s} appears in {int(row['n_phases'])} phases")

print()
print(f"Topics confined to a single phase (phase-specific): {len(phase_specific)}")
for _, row in phase_specific.iterrows():
    print(f"  {row['topic_label']:40s}")

print()
print("---")
print("Dominant topic per phase:")
for phase in PHASE_ORDER:
    phase_data = topic_phases[topic_phases["phase"] == phase]
    if len(phase_data) == 0:
        continue
    dominant = phase_data["topic_label"].value_counts().idxmax()
    count = phase_data["topic_label"].value_counts().max()
    pct = count / len(phase_data) * 100
    print(f"  {PHASE_DISPLAY[phase]:30s} \"{dominant}\"  ({pct:.0f}% of labeled comments)")

---
## 6. Additional Analyses

Beyond the phase-level summaries above, we examine sentiment across phase boundaries to quantify how much each new crisis revelation moved public sentiment.

### 6a. Phase Boundary Sentiment Shifts

For each adjacent phase transition, we compare sentiment in the 3 days before the boundary vs. the 3 days after. This quantifies how much each new revelation actually moved public sentiment. A Mann-Whitney U test determines whether the shift is statistically significant.

In [ ]:
from scipy.stats import mannwhitneyu

boundaries = []
for i in range(len(PHASE_ORDER) - 1):
    phase_a = PHASE_ORDER[i]
    phase_b = PHASE_ORDER[i + 1]

    boundary_date = df[df["phase"] == phase_b]["create_time"].min()
    label_a = PHASE_DISPLAY[phase_a]
    label_b = PHASE_DISPLAY[phase_b]

    window_start = boundary_date - pd.Timedelta(days=3)
    window_end = boundary_date + pd.Timedelta(days=3)

    before = df[(df["create_time"] >= window_start) & (df["create_time"] < boundary_date)]
    after = df[(df["create_time"] >= boundary_date) & (df["create_time"] <= window_end)]

    if len(before) == 0 or len(after) == 0:
        continue

    def polarity(subset):
        neg = (subset["sentiment_label"] == "NEGATIVE").sum()
        pos = (subset["sentiment_label"] == "POSITIVE").sum()
        total = len(subset)
        return (pos - neg) / total * 100 if total else 0

    pol_before = polarity(before)
    pol_after = polarity(after)
    pct_neg_before = (before["sentiment_label"] == "NEGATIVE").sum() / len(before) * 100
    pct_neg_after = (after["sentiment_label"] == "NEGATIVE").sum() / len(after) * 100

    scores_before = before["sentiment_score"].values
    scores_after = after["sentiment_score"].values
    u_stat, p_value = mannwhitneyu(scores_before, scores_after, alternative="two-sided")

    boundaries.append({
        "transition": f"{label_a}  >>>  {label_b}",
        "boundary_date": boundary_date,
        "n_before": len(before),
        "n_after": len(after),
        "pol_before": pol_before,
        "pol_after": pol_after,
        "delta_pol": pol_after - pol_before,
        "pct_neg_before": pct_neg_before,
        "pct_neg_after": pct_neg_after,
        "delta_neg": pct_neg_after - pct_neg_before,
        "u_stat": u_stat,
        "p_value": p_value,
    })

boundary_df = pd.DataFrame(boundaries)

print("Phase boundary sentiment shifts (3d before vs 3d after):")
print()
display(
    boundary_df[["transition", "n_before", "n_after", "pol_before", "pol_after",
                 "delta_pol", "delta_neg", "p_value"]].round(2)
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

y_pos = np.arange(len(boundary_df))

for i, (_, row) in enumerate(boundary_df.iterrows()):
    ax.plot([row["pol_before"], row["pol_after"]], [i, i],
            color="gray", linewidth=1.5, zorder=1)
    ax.plot(row["pol_before"], i, marker="o", color="steelblue",
            markersize=10, label="Before" if i == 0 else "", zorder=2)
    ax.plot(row["pol_after"], i, marker="s", color="crimson",
            markersize=10, label="After" if i == 0 else "", zorder=2)

    delta = row["delta_pol"]
    mid = (row["pol_before"] + row["pol_after"]) / 2
    ax.annotate(f"{delta:+.1f}",
                xy=(mid, i), xytext=(0, 12), textcoords="offset points",
                ha="center", fontsize=8, fontweight="bold",
                color="darkgreen" if delta > 0 else "darkred")

ax.set_yticks(y_pos)
ax.set_yticklabels(boundary_df["transition"], fontsize=8)
ax.axvline(x=0, color="black", linewidth=0.5)
ax.set_xlabel("Polarity Index")
ax.set_title("Sentiment Shift at Each Phase Boundary"
             "\n(before = 3d prior, after = 3d post)")
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.3)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "summary_boundary_shifts_forest.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
boundary_df["abs_delta_pol"] = boundary_df["delta_pol"].abs()
ranked = boundary_df.sort_values("abs_delta_pol", ascending=False).reset_index(drop=True)

print("Phase boundaries ranked by sentiment shift magnitude:")
print()
for _, row in ranked.iterrows():
    direction = "more negative" if row['delta_pol'] < 0 else "more positive"
    print(f"  {row['transition']}")
    print(f"    Polarity shift: {row['delta_pol']:+.1f} ({direction})")
    print(f"    %Neg shift:     {row['delta_neg']:+.1f}pp "
          f"(before: {row['pct_neg_before']:.1f}% -> after: {row['pct_neg_after']:.1f}%)")
    if pd.notna(row['p_value']):
        print(f"    Mann-Whitney:   U={row['u_stat']:.0f}, p={row['p_value']:.4f}")
    print()

top = ranked.iloc[0]
print(f"Biggest shift: {top['transition']}")
print(f"  Polarity {top['pol_before']:.1f} -> {top['pol_after']:.1f} "
      f"(delta {top['delta_pol']:+.1f})")

---
## 7. Key Takeaways

1. **Consistently negative sentiment across all phases.** The volume-weighted average polarity is −66.6, with every phase below −20. The Resignation Press Con phase was the most negative (79.0% neg), while the UAAP Ban phase was the least negative (53.0% neg).

2. **No significant polarizing or calming trend.** The Mann-Kendall test shows no monotonic trend — sentiment fluctuates from phase to phase rather than consistently improving or worsening over time.

3. **Outrage does not decay within phases.** Only 1 of 9 phases (Townhall Podcast) shows a statistically significant Day 1 vs Rest difference, and the effect is small (Cramer's V = 0.051). For all other phases, initial outrage neither decays nor intensifies significantly.

4. **Topics are largely phase-specific.** Most topics are confined to a single crisis phase — the conversation resets at each new revelation. Only a few themes (e.g., calls for justice, negligence allegations) persist across boundaries.

5. **Negligence and accountability topics carry the most negative sentiment.** Topics related to "Training Negligence Allegations," "CPR Cessation Controversy," and "Open Water Training Risks" have polarity scores below −75, reflecting concentrated outrage at systemic failures.